# CUDA exact top-k cosine similarity search
Runtime → Change runtime type → **T4 GPU**, then run all cells.

Args: `N d Q k batch`. If you switch GPU, change `-arch` (T4 `sm_75`, L4 `sm_89`, A100 `sm_80`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
%%writefile cuda_topk.cu
// =============================================================================
//  cuda_topk.cu  —  CUDA (GPU)
//  Exact top-k cosine similarity search:
//    (1) normalise kernel       : one block per row, warp-shuffle reduction
//    (2) tiled similarity kernel: shared-memory tiles, S = Qn * Dn^T
//    (3) hierarchical top-k     :
//          per-thread local top-k (sorted list in registers/local memory)
//          -> warp-level merge  (__shfl_down_sync argmax)
//          -> block-level merge (argmax across warps, k rounds)
//          -> global merge kernel (one block per query merges all block results)
//
//  Build (Colab T4): nvcc -O3 -arch=sm_75 cuda_topk.cu -o cuda_topk
//        (L4: sm_89, A100: sm_80, V100: sm_70)
//  Run  : ./cuda_topk  N  d  Q  k  [batch]
//         ./cuda_topk 1000000 768 256 10 64
// =============================================================================
#include <cuda_runtime.h>

#include <algorithm>
#include <climits>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t e_ = (call);                                                  \
        if (e_ != cudaSuccess) {                                                  \
            fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e_),   \
                    __FILE__, __LINE__);                                          \
            exit(1);                                                              \
        }                                                                         \
    } while (0)

constexpr int TILE          = 16;    // similarity tile size (16x16 threads)
constexpr int MAX_K         = 128;   // largest k supported
constexpr int TOPK_THREADS  = 256;   // threads per top-k block
constexpr int ITEMS_PER_THR = 32;    // target scores scanned per thread in stage 1
constexpr unsigned FULL     = 0xffffffffu;

// ---------- deterministic RNG (identical in serial / OpenMP / CUDA) ----------
static inline uint64_t splitmix64(uint64_t &s) {
    uint64_t z = (s += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
static void fill_random(std::vector<float> &v, uint64_t seed) {
    uint64_t s = seed;
    for (auto &x : v) x = (float)((splitmix64(s) >> 40) * (1.0 / 16777216.0)) * 2.0f - 1.0f;
}

// =============================================================================
// (1) Normalisation: one block per row
// =============================================================================
__inline__ __device__ float warp_sum(float v) {
    for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(FULL, v, off);
    return v;
}

__global__ void normalize_kernel(float *X, int rows, int d) {
    int r = blockIdx.x;
    if (r >= rows) return;
    float *row = X + (size_t)r * d;

    float s = 0.0f;
    for (int j = threadIdx.x; j < d; j += blockDim.x) s += row[j] * row[j];

    __shared__ float warp_part[32];
    __shared__ float inv;
    int lane = threadIdx.x & 31, warp = threadIdx.x >> 5;
    s = warp_sum(s);
    if (lane == 0) warp_part[warp] = s;
    __syncthreads();
    if (warp == 0) {
        s = lane < (blockDim.x + 31) / 32 ? warp_part[lane] : 0.0f;
        s = warp_sum(s);
        if (lane == 0) inv = s > 0.0f ? 1.0f / sqrtf(s) : 0.0f;
    }
    __syncthreads();
    for (int j = threadIdx.x; j < d; j += blockDim.x) row[j] *= inv;
}

// =============================================================================
// (2) Tiled similarity: S[q][n] = sum_j Q[q][j] * D[n][j]
//     Block (16x16) computes a 16-query x 16-vector tile of S.
//     Q and D tiles are staged in shared memory, so every element loaded from
//     global memory is reused 16 times. Ds is padded (+1) to avoid bank conflicts.
// =============================================================================
__global__ void similarity_kernel(const float *__restrict__ Qm, const float *__restrict__ D,
                                  float *__restrict__ S, int nq, int N, int d) {
    __shared__ float Qs[TILE][TILE];
    __shared__ float Ds[TILE][TILE + 1];

    int tx = threadIdx.x, ty = threadIdx.y;
    int q = blockIdx.y * TILE + ty;          // query row this thread outputs
    int colBase = blockIdx.x * TILE;
    int n = colBase + tx;                    // db vector this thread outputs
    int drow = colBase + ty;                 // db row this thread helps load

    float acc = 0.0f;
    for (int t = 0; t < d; t += TILE) {
        Qs[ty][tx] = (q < nq && t + tx < d) ? Qm[(size_t)q * d + t + tx] : 0.0f;
        Ds[ty][tx] = (drow < N && t + tx < d) ? D[(size_t)drow * d + t + tx] : 0.0f;
        __syncthreads();
#pragma unroll
        for (int i = 0; i < TILE; ++i) acc += Qs[ty][i] * Ds[tx][i];
        __syncthreads();
    }
    if (q < nq && n < N) S[(size_t)q * N + n] = acc;
}

// =============================================================================
// (3) Hierarchical top-k
// =============================================================================
// higher score wins; ties -> smaller index (deterministic, matches CPU code)
__device__ __forceinline__ bool better(float a, int ia, float b, int ib) {
    return a > b || (a == b && ia < ib);
}

// Level 1: per-thread sorted list (descending), insertion sort, size <= k
__device__ __forceinline__ void local_insert(float *vals, int *ids, int &cnt, int k, float v, int id) {
    if (cnt == k && !better(v, id, vals[k - 1], ids[k - 1])) return;   // fast reject
    int pos = (cnt < k) ? cnt++ : k - 1;
    while (pos > 0 && better(v, id, vals[pos - 1], ids[pos - 1])) {
        vals[pos] = vals[pos - 1];
        ids[pos] = ids[pos - 1];
        --pos;
    }
    vals[pos] = v;
    ids[pos] = id;
}

// Level 2: warp argmax via shuffles; carries (value, index, owner thread)
__device__ __forceinline__ void warp_best(float &v, int &id, int &owner) {
    for (int off = 16; off > 0; off >>= 1) {
        float ov = __shfl_down_sync(FULL, v, off);
        int oid  = __shfl_down_sync(FULL, id, off);
        int oown = __shfl_down_sync(FULL, owner, off);
        if (better(ov, oid, v, id)) { v = ov; id = oid; owner = oown; }
    }
}

// Level 3: block merge. Each thread's list is sorted, so its head is its best.
// Repeat k times: find the best head in the block (warp argmax -> block argmax),
// emit it, and the owning thread advances its head.
__device__ void block_select_k(const float *vals, const int *ids, int cnt, int k,
                               float *outV, int *outI) {
    __shared__ float sV[32];
    __shared__ int sI[32], sO[32];
    __shared__ int winner;
    int lane = threadIdx.x & 31, warp = threadIdx.x >> 5;
    int nwarps = (blockDim.x + 31) / 32;
    int head = 0;

    for (int r = 0; r < k; ++r) {
        float v = head < cnt ? vals[head] : -INFINITY;
        int id  = head < cnt ? ids[head]  : INT_MAX;
        int own = threadIdx.x;
        warp_best(v, id, own);
        if (lane == 0) { sV[warp] = v; sI[warp] = id; sO[warp] = own; }
        __syncthreads();
        if (warp == 0) {
            v   = lane < nwarps ? sV[lane] : -INFINITY;
            id  = lane < nwarps ? sI[lane] : INT_MAX;
            own = lane < nwarps ? sO[lane] : -1;
            warp_best(v, id, own);
            if (lane == 0) {
                outV[r] = v;
                outI[r] = (v == -INFINITY) ? -1 : id;
                winner = (v == -INFINITY) ? -1 : own;
            }
        }
        __syncthreads();
        if (threadIdx.x == winner) ++head;
    }
}

// Stage 1: grid (blocksPerQuery, nq). Each block scans a contiguous segment of
// one query's score row and writes its block-level top-k into cand[].
__global__ void topk_stage1(const float *__restrict__ S, int N, int k, int blocksPerQuery,
                            float *candV, int *candI) {
    int q = blockIdx.y;
    int seg = (N + blocksPerQuery - 1) / blocksPerQuery;
    int lo = blockIdx.x * seg, hi = min(N, lo + seg);
    const float *row = S + (size_t)q * N;

    float vals[MAX_K];
    int ids[MAX_K];
    int cnt = 0;
    for (int i = lo + threadIdx.x; i < hi; i += blockDim.x)          // coalesced reads
        local_insert(vals, ids, cnt, k, row[i], i);

    size_t base = ((size_t)q * blocksPerQuery + blockIdx.x) * k;
    block_select_k(vals, ids, cnt, k, candV + base, candI + base);
}

// Stage 2 (global merge): one block per query merges blocksPerQuery*k candidates.
__global__ void topk_stage2(const float *candV, const int *candI, int nCand, int k,
                            float *outV, int *outI) {
    int q = blockIdx.x;
    const float *cv = candV + (size_t)q * nCand;
    const int *ci   = candI + (size_t)q * nCand;

    float vals[MAX_K];
    int ids[MAX_K];
    int cnt = 0;
    for (int i = threadIdx.x; i < nCand; i += blockDim.x)
        if (ci[i] >= 0) local_insert(vals, ids, cnt, k, cv[i], ci[i]);

    block_select_k(vals, ids, cnt, k, outV + (size_t)q * k, outI + (size_t)q * k);
}

// =============================================================================
// Host
// =============================================================================
int main(int argc, char **argv) {
    int N     = argc > 1 ? atoi(argv[1]) : 1000000;
    int d     = argc > 2 ? atoi(argv[2]) : 768;
    int Q     = argc > 3 ? atoi(argv[3]) : 256;
    int k     = argc > 4 ? atoi(argv[4]) : 10;
    int batch = argc > 5 ? atoi(argv[5]) : 64;
    if (k > MAX_K) { fprintf(stderr, "k must be <= %d\n", MAX_K); return 1; }
    if (k > N) k = N;
    batch = std::min(batch, Q);

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("=== CUDA | %s | N=%d d=%d Q=%d k=%d batch=%d ===\n", prop.name, N, d, Q, k, batch);

    // ---- host data (same generator + seeds as the CPU versions) ----
    std::vector<float> h_db((size_t)N * d), h_q((size_t)Q * d);
    fill_random(h_db, 42);
    fill_random(h_q, 7);

    int blocksPerQuery = std::max(1, std::min(1024, (N + TOPK_THREADS * ITEMS_PER_THR - 1) /
                                                        (TOPK_THREADS * ITEMS_PER_THR)));
    int nCand = blocksPerQuery * k;

    size_t bytes_db = (size_t)N * d * sizeof(float);
    size_t bytes_S = (size_t)batch * N * sizeof(float);
    size_t bytes_c = (size_t)batch * nCand * (sizeof(float) + sizeof(int));
    size_t freeB, totalB;
    CUDA_CHECK(cudaMemGetInfo(&freeB, &totalB));
    double needGB = (bytes_db + (size_t)Q * d * 4 + bytes_S + bytes_c + (size_t)Q * k * 8) / 1e9;
    printf("GPU memory  : need %.2f GB, free %.2f GB\n", needGB, freeB / 1e9);
    if (needGB * 1e9 > freeB * 0.95) {
        fprintf(stderr, "Not enough GPU memory: reduce N or batch (chunking over N would be needed).\n");
        return 1;
    }

    float *d_db, *d_q, *d_S, *d_cV, *d_outV;
    int *d_cI, *d_outI;
    CUDA_CHECK(cudaMalloc(&d_db, bytes_db));
    CUDA_CHECK(cudaMalloc(&d_q, (size_t)Q * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_S, bytes_S));
    CUDA_CHECK(cudaMalloc(&d_cV, (size_t)batch * nCand * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_cI, (size_t)batch * nCand * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_outV, (size_t)Q * k * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_outI, (size_t)Q * k * sizeof(int)));

    cudaEvent_t e0, e1;
    CUDA_CHECK(cudaEventCreate(&e0));
    CUDA_CHECK(cudaEventCreate(&e1));
    auto elapsed = [&]() { float ms; CUDA_CHECK(cudaEventSynchronize(e1)); CUDA_CHECK(cudaEventElapsedTime(&ms, e0, e1)); return ms / 1000.0; };

    // warm-up launch (first launch includes CUDA context / module load cost)
    normalize_kernel<<<1, 256>>>(d_q, 0, d);
    CUDA_CHECK(cudaDeviceSynchronize());

    // ---- host -> device ----
    cudaEventRecord(e0);
    CUDA_CHECK(cudaMemcpy(d_db, h_db.data(), bytes_db, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_q, h_q.data(), (size_t)Q * d * sizeof(float), cudaMemcpyHostToDevice));
    cudaEventRecord(e1);
    double t_h2d = elapsed();

    // ---- normalise ----
    cudaEventRecord(e0);
    normalize_kernel<<<N, 256>>>(d_db, N, d);
    normalize_kernel<<<Q, 256>>>(d_q, Q, d);
    cudaEventRecord(e1);
    CUDA_CHECK(cudaGetLastError());
    double t_norm = elapsed();

    // ---- search, batch by batch ----
    double t_sim = 0.0, t_topk = 0.0;
    for (int q0 = 0; q0 < Q; q0 += batch) {
        int nb = std::min(batch, Q - q0);

        dim3 sb(TILE, TILE), sg((N + TILE - 1) / TILE, (nb + TILE - 1) / TILE);
        cudaEventRecord(e0);
        similarity_kernel<<<sg, sb>>>(d_q + (size_t)q0 * d, d_db, d_S, nb, N, d);
        cudaEventRecord(e1);
        CUDA_CHECK(cudaGetLastError());
        t_sim += elapsed();

        cudaEventRecord(e0);
        topk_stage1<<<dim3(blocksPerQuery, nb), TOPK_THREADS>>>(d_S, N, k, blocksPerQuery, d_cV, d_cI);
        topk_stage2<<<nb, TOPK_THREADS>>>(d_cV, d_cI, nCand, k,
                                           d_outV + (size_t)q0 * k, d_outI + (size_t)q0 * k);
        cudaEventRecord(e1);
        CUDA_CHECK(cudaGetLastError());
        t_topk += elapsed();
    }

    // ---- device -> host ----
    std::vector<float> h_outV((size_t)Q * k);
    std::vector<int> h_outI((size_t)Q * k);
    cudaEventRecord(e0);
    CUDA_CHECK(cudaMemcpy(h_outV.data(), d_outV, h_outV.size() * sizeof(float), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_outI.data(), d_outI, h_outI.size() * sizeof(int), cudaMemcpyDeviceToHost));
    cudaEventRecord(e1);
    double t_d2h = elapsed();

    // ---- correctness check on CPU (first few queries, double precision) ----
    int nverify = std::min(Q, 4);
    double recall = 0.0, maxErr = 0.0;
    std::vector<double> dbNorm(N);
    for (int i = 0; i < N; ++i) {
        double s = 0; for (int j = 0; j < d; ++j) s += (double)h_db[(size_t)i * d + j] * h_db[(size_t)i * d + j];
        dbNorm[i] = std::sqrt(s);
    }
    std::vector<std::pair<double, int>> ref(N);
    for (int q = 0; q < nverify; ++q) {
        const float *qv = h_q.data() + (size_t)q * d;
        double qn = 0; for (int j = 0; j < d; ++j) qn += (double)qv[j] * qv[j];
        qn = std::sqrt(qn);
        for (int i = 0; i < N; ++i) {
            double s = 0; const float *x = h_db.data() + (size_t)i * d;
            for (int j = 0; j < d; ++j) s += (double)qv[j] * x[j];
            ref[i] = {s / (qn * dbNorm[i]), i};
        }
        std::partial_sort(ref.begin(), ref.begin() + k, ref.end(),
                          [](const std::pair<double, int> &a, const std::pair<double, int> &b) {
                              return a.first > b.first || (a.first == b.first && a.second < b.second); });
        int hits = 0;
        for (int j = 0; j < k; ++j) {
            maxErr = std::max(maxErr, std::fabs(ref[j].first - h_outV[(size_t)q * k + j]));
            for (int r = 0; r < k; ++r) if (h_outI[(size_t)q * k + j] == ref[r].second) { ++hits; break; }
        }
        recall += (double)hits / k;
    }
    recall /= nverify;

    double search = t_sim + t_topk;
    double total = t_h2d + t_norm + search + t_d2h;
    double checksum = 0.0;
    for (float v : h_outV) checksum += v;

    printf("H2D copy    : %10.4f s\n", t_h2d);
    printf("normalise   : %10.4f s\n", t_norm);
    printf("similarity  : %10.4f s   (%.2f GFLOP/s, %.1f GB/s db reads)\n", t_sim,
           2.0 * Q * (double)N * d / t_sim / 1e9,
           (double)bytes_db * ((Q + batch - 1) / batch) * ((batch + TILE - 1) / TILE) / t_sim / 1e9);
    printf("top-k       : %10.4f s   (%.1f%% of search time, %d blocks/query)\n", t_topk,
           100.0 * t_topk / search, blocksPerQuery);
    printf("D2H copy    : %10.4f s\n", t_d2h);
    printf("TOTAL       : %10.4f s   (kernels only: %.4f s)\n", total, t_norm + search);
    printf("throughput  : %10.1f queries/s\n", Q / search);
    printf("recall@%d    : %.4f   max |score err| = %.2e   (vs CPU double reference, %d queries)\n",
           k, recall, maxErr, nverify);
    printf("checksum    : %.6f   (sum of all top-k scores; compare across versions)\n", checksum);
    printf("query 0 top-%d:", std::min(k, 5));
    for (int j = 0; j < std::min(k, 5); ++j) printf("  [%d] %.4f", h_outI[j], h_outV[j]);
    printf("\n");
    printf("CSV,cuda,%d,%d,%d,%d,%d,%.6f,%.6f,%.6f,%.6f\n", N, d, Q, k, batch,
           t_norm, t_sim, t_topk, total);

    cudaFree(d_db); cudaFree(d_q); cudaFree(d_S); cudaFree(d_cV); cudaFree(d_cI);
    cudaFree(d_outV); cudaFree(d_outI);
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 cuda_topk.cu -o cuda_topk

## 1. Single run (same sizes as the CPU runs, for direct comparison)

In [ ]:
!./cuda_topk 200000 384 64 10 32

## 2. Large run (the abstract's 1M x 768 case)

In [ ]:
!./cuda_topk 1000000 768 256 10 64

## 3. Sweeps: N, d, k. Collects the CSV lines.

In [ ]:
import subprocess, pandas as pd
cols = ["impl","N","d","Q","k","batch","t_norm","t_sim","t_topk","t_total"]
def run(N, d, Q, k, batch=64):
    out = subprocess.run(["./cuda_topk", str(N), str(d), str(Q), str(k), str(batch)],
                         capture_output=True, text=True).stdout
    line = [l for l in out.splitlines() if l.startswith("CSV,")][0]
    return line.split(",")[1:]

rows = []
for N in [100_000, 250_000, 500_000, 1_000_000]:
    rows.append(run(N, 768, 256, 10))
for d in [128, 384, 768, 1536]:
    rows.append(run(500_000, d, 256, 10))
for k in [1, 10, 32, 64, 128]:
    rows.append(run(1_000_000, 768, 256, k))
df = pd.DataFrame(rows, columns=cols)
for c in cols[1:]: df[c] = pd.to_numeric(df[c])
df["topk_share_%"] = 100 * df.t_topk / (df.t_sim + df.t_topk)
df.to_csv("cuda_results.csv", index=False)
df

In [ ]:
import matplotlib.pyplot as plt
ks = df[(df.N == 1_000_000) & (df.d == 768)].drop_duplicates("k").sort_values("k")
plt.figure(figsize=(6,4))
plt.bar(ks.k.astype(str), ks.t_sim, label="similarity")
plt.bar(ks.k.astype(str), ks.t_topk, bottom=ks.t_sim, label="top-k")
plt.xlabel("k"); plt.ylabel("time (s)"); plt.title("GPU time split vs k (N=1M, d=768, Q=256)")
plt.legend(); plt.tight_layout(); plt.show()